# Unit 2 Assignment: Mixture of Experts (MoE) Router

## Smart Customer Support Router using Groq API
### Done by : Suchitra Shankar, PES2UG23CS608
This notebook implements a Mixture of Experts (MoE) architecture for routing customer support queries to specialized AI experts.

Experts implemented:
- Technical Expert
- Billing Expert
- Sales Expert
- General Support Expert

The system includes:
- An LLM-based Router (intent classifier)
- Expert-specific system prompts
- An Orchestrator that dispatches queries

## Step 1: Environment Setup

We install required dependencies:
- `groq` for API access
- `python-dotenv` (optional for environment management)

The Groq API key is securely loaded using `getpass()`.

In [45]:
!pip install groq python-dotenv

In [46]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")

Enter your Groq API Key: ··········


In [47]:
from groq import Groq
import os

client = Groq(api_key=os.environ["GROQ_API_KEY"])

models = client.models.list()
for m in models.data:
    print(m.id)

meta-llama/llama-4-maverick-17b-128e-instruct
meta-llama/llama-4-scout-17b-16e-instruct
openai/gpt-oss-20b
groq/compound
whisper-large-v3
canopylabs/orpheus-arabic-saudi
moonshotai/kimi-k2-instruct
meta-llama/llama-guard-4-12b
openai/gpt-oss-safeguard-20b
llama-3.1-8b-instant
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-120b
moonshotai/kimi-k2-instruct-0905
canopylabs/orpheus-v1-english
qwen/qwen3-32b
whisper-large-v3-turbo
groq/compound-mini
meta-llama/llama-prompt-guard-2-86m
llama-3.3-70b-versatile
allam-2-7b


## Step 2: Model Configuration

We use a currently supported Groq model:

- `llama-3.1-8b-instant`

This model is used for:
- Routing (temperature = 0)
- Expert generation (temperature varies by expert)

In production systems, routing and generation models may be separated.

In [48]:
!pip install groq
from groq import Groq
import os

client = Groq(api_key=os.environ["GROQ_API_KEY"])

# IMPORTANT: Replace "YOUR_GROQ_MODEL_HERE" with a currently supported model from Groq's documentation.
# Check https://console.groq.com/docs/deprecations for the latest available models.
BASE_MODEL = "llama-3.1-8b-instant"


## Step 3: Define Experts (MoE Configuration)

Each expert is simulated using a different **system prompt**.

### Experts:
- **Technical Expert**
  - Handles debugging, errors, code issues
  - Structured step-by-step responses

- **Billing Expert**
  - Handles refunds, charges, invoices
  - Empathetic and policy-driven

- **Sales Expert**
  - Handles pricing and feature inquiries
  - Highlights value and encourages next steps

- **General Expert**
  - Handles greetings and non-specific queries

Each expert uses a different temperature setting to reflect tone and flexibility.

In [49]:

# -----------------------------
# Experts
# -----------------------------

MODEL_CONFIG = {
    "technical": {
        "system_prompt": (
            "You are a senior technical support engineer.\n\n"
            "Your responsibilities:\n"
            "- Diagnose software bugs and technical errors.\n"
            "- Provide step-by-step debugging guidance.\n"
            "- Include code snippets when helpful.\n"
            "- Be precise, structured, and concise.\n"
            "- Avoid unnecessary explanations or marketing language.\n\n"
            "Response Format:\n"
            "1. Brief explanation of the issue\n"
            "2. Likely cause\n"
            "3. Step-by-step solution\n"
            "4. Example code (if applicable)\n\n"
            "If information is missing, ask a clarifying question before assuming."
        ),
        "temperature": 0.6
    },

    "billing": {
        "system_prompt": (
            "You are a customer billing specialist.\n\n"
            "Your responsibilities:\n"
            "- Handle refunds, duplicate charges, subscriptions, invoices, and payments.\n"
            "- Be empathetic and professional.\n"
            "- Clearly explain relevant policies.\n"
            "- Provide actionable next steps.\n\n"
            "Response Guidelines:\n"
            "- Acknowledge the customer's concern.\n"
            "- Explain what may have happened.\n"
            "- Provide resolution steps.\n"
            "- Avoid technical jargon.\n"
            "- Never blame the customer.\n\n"
            "Maintain a calm and reassuring tone."
        ),
        "temperature": 0.7
    },

    "sales": {
        "system_prompt": (
            "You are a knowledgeable and persuasive sales consultant.\n\n"
            "Your responsibilities:\n"
            "- Answer product, pricing, feature, and plan inquiries.\n"
            "- Highlight value propositions and competitive advantages.\n"
            "- Match product features to the customer's needs.\n"
            "- Encourage conversion without being pushy.\n\n"
            "Response Guidelines:\n"
            "- Clearly explain available plans or features.\n"
            "- Emphasize benefits, not just features.\n"
            "- Offer next steps (demo, signup, contact sales).\n"
            "- Maintain a confident and helpful tone.\n\n"
            "Do NOT handle refunds or technical debugging."
        ),
        "temperature": 0.8
    },

    "general": {
        "system_prompt": (
            "You are a friendly and helpful customer support assistant.\n\n"
            "Your responsibilities:\n"
            "- Handle general inquiries and casual conversation.\n"
            "- Provide simple and clear explanations.\n"
            "- Redirect to the appropriate department when necessary.\n\n"
            "Maintain a polite, warm, and concise tone."
        ),
        "temperature": 0.7
    }
}

## Step 4: Router (Intent Classifier)

The router:
- Uses temperature = 0 for deterministic classification
- Classifies input into:
  - technical
  - billing
  - sales
  - general
- Returns only the category name

The router ensures queries are dispatched to the correct expert.

In [50]:
def route_prompt(user_input: str) -> str:
    routing_prompt = f"""
Classify the user's message into exactly one of these categories:
technical, billing, sales, general

Return ONLY one lowercase word.
No quotes.
No punctuation.
No explanation.

Message: "{user_input}"
"""

    response = client.chat.completions.create(
        model=BASE_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": "You are a strict intent classifier."},
            {"role": "user", "content": routing_prompt}
        ]
    )

    raw_output = response.choices[0].message.content
    print("RAW MODEL OUTPUT:", repr(raw_output))  # 👈 ADD THIS

    category = raw_output.strip().lower()

    if category not in MODEL_CONFIG:
        category = "general"

    return category

## Step 5: Orchestrator

The `process_request()` function:

1. Calls the router to determine category
2. Selects the appropriate expert configuration
3. Sends the query to the LLM with the expert's system prompt
4. Returns the final response

This clean separation ensures modular and extensible architecture.

In [51]:

# -----------------------------
# Orchestrator
# -----------------------------

def process_request(user_input: str) -> str:
    category = route_prompt(user_input)
    expert = MODEL_CONFIG[category]

    response = client.chat.completions.create(
        model=BASE_MODEL,
        temperature=expert["temperature"],
        messages=[
            {"role": "system", "content": expert["system_prompt"]},
            {"role": "user", "content": user_input}
        ]
    )

    return response.choices[0].message.content



## Step 6: Testing & Evaluation

We test the router using multiple query groups:

- Technical test cases
- Billing test cases
- Sales test cases
- General test cases
- Edge cases

This helps evaluate routing accuracy and expert specialization.

In [52]:

# -----------------------------
# Test
# -----------------------------

query = "My python script is throwing an IndexError."
print("Category:", route_prompt(query))
print("\nResponse:\n", process_request(query))

RAW MODEL OUTPUT: 'technical'
Category: technical
RAW MODEL OUTPUT: 'technical'

Response:
 **Issue:** Python script is throwing an IndexError.
**Likely Cause:** The error occurs when you're trying to access an element in a list or a string using an index that's out of range.

**Step-by-Step Solution:**

1. **Check the index**: Verify that the index you're using is within the valid range. You can do this by checking the length of the list or string:
   ```python
my_list = [1, 2, 3]
index = 3
if index < len(my_list):
    print(my_list[index])
else:
    print("Index out of range")
```

2. **Verify list/string indexing**: Make sure you're not trying to access an element using a negative index or an index that's greater than or equal to the length of the list or string.

3. **Use try-except block**: Wrap the code that's causing the IndexError in a try-except block to catch the exception and provide a custom error message:
   ```python
try:
    my_list = [1, 2, 3]
    index = 3
    print(my

In [53]:
technical_tests = [
    "My API returns a 500 error after deployment.",
    "I'm getting a segmentation fault in my C++ program.",
    "Why is my SQL query timing out?",
    "The app crashes when I click the login button.",
    "How do I fix a KeyError in Python?"
]

for q in technical_tests:
    print("\nUser:", q)
    print("→ Routed To:", route_prompt(q))


User: My API returns a 500 error after deployment.
RAW MODEL OUTPUT: 'technical'
→ Routed To: technical

User: I'm getting a segmentation fault in my C++ program.
RAW MODEL OUTPUT: 'technical'
→ Routed To: technical

User: Why is my SQL query timing out?
RAW MODEL OUTPUT: 'technical'
→ Routed To: technical

User: The app crashes when I click the login button.
RAW MODEL OUTPUT: 'technical'
→ Routed To: technical

User: How do I fix a KeyError in Python?
RAW MODEL OUTPUT: 'technical'
→ Routed To: technical


In [54]:
billing_tests = [
    "I was charged twice this month.",
    "Can I get a refund for last week’s payment?",
    "Why did my subscription renew automatically?",
    "My invoice shows an unexpected fee.",
    "How do I update my payment method?"
]

for q in billing_tests:
    print("\nUser:", q)
    print("→ Routed To:", route_prompt(q))


User: I was charged twice this month.
RAW MODEL OUTPUT: 'billing'
→ Routed To: billing

User: Can I get a refund for last week’s payment?
RAW MODEL OUTPUT: 'billing'
→ Routed To: billing

User: Why did my subscription renew automatically?
RAW MODEL OUTPUT: 'billing'
→ Routed To: billing

User: My invoice shows an unexpected fee.
RAW MODEL OUTPUT: 'billing'
→ Routed To: billing

User: How do I update my payment method?
RAW MODEL OUTPUT: 'billing'
→ Routed To: billing


In [55]:
sales_tests = [
    "Do you offer enterprise pricing?",
    "What features are included in the premium plan?",
    "Is there a discount for students?",
    "Can I schedule a product demo?",
    "How does your product compare to competitors?"
]

for q in sales_tests:
    print("\nUser:", q)
    print("→ Routed To:", route_prompt(q))


User: Do you offer enterprise pricing?
RAW MODEL OUTPUT: 'billing'
→ Routed To: billing

User: What features are included in the premium plan?
RAW MODEL OUTPUT: 'technical'
→ Routed To: technical

User: Is there a discount for students?
RAW MODEL OUTPUT: 'general'
→ Routed To: general

User: Can I schedule a product demo?
RAW MODEL OUTPUT: 'sales'
→ Routed To: sales

User: How does your product compare to competitors?
RAW MODEL OUTPUT: 'general'
→ Routed To: general


In [56]:
general_tests = [
    "Hello!",
    "Can you tell me more about your company?",
    "Thanks for your help.",
    "What does your company do?",
    "Good morning!"
]

for q in general_tests:
    print("\nUser:", q)
    print("→ Routed To:", route_prompt(q))


User: Hello!
RAW MODEL OUTPUT: 'general'
→ Routed To: general

User: Can you tell me more about your company?
RAW MODEL OUTPUT: 'general'
→ Routed To: general

User: Thanks for your help.
RAW MODEL OUTPUT: 'general'
→ Routed To: general

User: What does your company do?
RAW MODEL OUTPUT: 'general'
→ Routed To: general

User: Good morning!
RAW MODEL OUTPUT: 'general'
→ Routed To: general


In [57]:
edge_tests = [
    "I want to cancel my subscription.",
    "Your product isn't working for me.",
    "How much does it cost to upgrade?",
    "The payment page is showing an error.",
    "I need help immediately."
]

for q in edge_tests:
    print("\nUser:", q)
    print("→ Routed To:", route_prompt(q))


User: I want to cancel my subscription.
RAW MODEL OUTPUT: 'billing'
→ Routed To: billing

User: Your product isn't working for me.
RAW MODEL OUTPUT: 'technical'
→ Routed To: technical

User: How much does it cost to upgrade?
RAW MODEL OUTPUT: 'billing'
→ Routed To: billing

User: The payment page is showing an error.
RAW MODEL OUTPUT: 'billing'
→ Routed To: billing

User: I need help immediately.
RAW MODEL OUTPUT: 'general'
→ Routed To: general


## Architecture

This system implements a simplified Mixture of Experts (MoE) design:

User Input

     ↓

LLM Router (Intent Classification)

     ↓

Selected Expert Configuration

     ↓
LLM Response Generation
     
     ↓

Final Output

---

Key Features:
- Deterministic routing
- Specialized expert prompts
- Modular architecture
- Extensible for tool-based experts

This structure mirrors real-world AI support systems.